In [1]:
from personamem_mem0_prep import (
    load_personamem_jsonl_for_mem0,
    short_view_user_bundle,
)

In [2]:
bundles = load_personamem_jsonl_for_mem0(
    "exports/personamem_benchmark_text_32k_user_bundles.jsonl",
    split="benchmark_text",
)

In [3]:
print(short_view_user_bundle(bundles[0]))

user_id: 10
chat_history: 162 messages
benchmark_rows: 31
metadata:
  - total_messages: 162
  - final_token_count: 31516
first_message:
  role: system
  content: 'You are an AI assistant helping a user with the following persona:\n\n{\n  "short_persona": "a fellow officer in the Indian Air Force, who admires Amar Preet Singh\'s career.",\n  "name": "Squadron Leader '
last_message:
  role: assistant
  content: 'Certainly, Squadron Leader Rao. The Defence Institute of Physiology & Allied Sciences (DIPAS) conducted a 5-year longitudinal comparative study involving 120 long-serving personnel across various bran'


In [4]:
bundles_multimodal = load_personamem_jsonl_for_mem0(
    "exports/personamem_benchmark_multimodal_32k_user_bundles.jsonl",
    split="benchmark_multimodal",
)

In [5]:
print(short_view_user_bundle(bundles_multimodal[0]))

user_id: 10
chat_history: 162 messages
benchmark_rows: 31
metadata:
  - total_messages: 162
  - final_token_count: 31516
first_message:
  role: system
  content: 'You are an AI assistant helping a user with the following persona:\n\n{\n  "short_persona": "a fellow officer in the Indian Air Force, who admires Amar Preet Singh\'s career.",\n  "name": "Squadron Leader '
last_message:
  role: assistant
  content: 'Certainly, Squadron Leader Rao. The Defence Institute of Physiology & Allied Sciences (DIPAS) conducted a 5-year longitudinal comparative study involving 120 long-serving personnel across various bran'


In [11]:
# Look at messages with images in content (multimodal: content can be a list of text + image_url parts)
def messages_with_images(chat_history):
    """Yield (index, message) for messages where content is a list containing an image part."""
    for i, msg in enumerate(chat_history):
        c = msg.get("content")
        if not isinstance(c, list):
            continue
        for part in c:
            if isinstance(part, dict) and part.get("type") == "image_url":
                yield i, msg
                break

# Check first bundle's chat_history for image messages
chat = bundles_multimodal[1].chat_history

with_images = list(messages_with_images(chat))

print(f"Bundle 0: {len(chat)} total messages, {len(with_images)} message(s) with images")
for idx, msg in with_images[:5]:  # show first 5
    text_parts = [p.get("text", "")[:80] for p in msg["content"] if isinstance(p, dict) and p.get("type") == "text"]
    print(f"  Message {idx} (role={msg['role']}): text preview: {text_parts}")
if not with_images:
    print("  (This bundle has no image messages; try another bundle index or persona.)")

Bundle 0: 192 total messages, 2 message(s) with images
  Message 142 (role=user): text preview: ['In this picture I took while we were outside, what factors could be influencing ']
  Message 144 (role=user): text preview: ['In this shot I captured on my commute, what small details about their posture an']


In [ ]:
# --- How images are stored & how to parse them ---
# Structure: messages with images have content = list of parts. Each part is a dict:
#   - {"type": "text", "text": "..."}
#   - {"type": "image_url", "image_url": <url>}   where <url> is EITHER:
#       • a string: "data:image/jpeg;base64,/9j/4AAQ..."  (inline base64)
#       • or a dict: {"url": "data:image/jpeg;base64,..."}
# So far in PersonaMem they appear to all use the same format: data URL with base64.

import base64
from pprint import pprint

def get_image_url_from_part(part):
    """Get the raw data URL string from an image_url part. Handles both str and dict."""
    if not isinstance(part, dict) or part.get("type") != "image_url":
        return None
    url = part.get("image_url")
    if isinstance(url, dict):
        url = url.get("url") or url.get("image_url")
    return url if isinstance(url, str) else None

def parse_data_url(data_url):
    """
    Parse a data:image/...;base64,<b64> string into (mime_type, raw_bytes).
    Returns (None, None) if not a valid image data URL.
    """
    if not data_url or not data_url.startswith("data:image"):
        return None, None
    if "," not in data_url:
        return None, None
    header, b64 = data_url.split(",", 1)
    # header is e.g. "data:image/jpeg;base64"
    mime = "image/jpeg"  # default
    if "image/" in header:
        mime = header.split("image/")[1].split(";")[0].strip() or mime
    try:
        return mime, base64.b64decode(b64)
    except Exception:
        return None, None

def extract_images_from_message(msg):
    """Yield (mime_type, raw_bytes) for every image in a single message."""
    c = msg.get("content")
    if not isinstance(c, list):
        return
    for part in c:
        url = get_image_url_from_part(part)
        if url:
            mime, raw = parse_data_url(url)
            if raw is not None:
                yield mime, raw

# --- Example: show raw structure of one message with image ---
if with_images:
    idx, example_msg = with_images[0]
    print("Example message structure (content parts only):")
    for j, p in enumerate(example_msg["content"]):
        if isinstance(p, dict):
            if p.get("type") == "text":
                print(f"  part[{j}]: type=text, len(text)={len(p.get('text',''))}")
            elif p.get("type") == "image_url":
                url = get_image_url_from_part(p)
                print(f"  part[{j}]: type=image_url, url_prefix={url[:60] if url else None}...")
    print()
    # Parse first image and show it's easy
    images = list(extract_images_from_message(example_msg))
    print(f"Parsed {len(images)} image(s) from that message. First: mime={images[0][0]}, size={len(images[0][1])} bytes")
    print("Conclusion: parsing is straightforward; all PersonaMem multimodal images use the same data URL + base64 format.")
else:
    print("No messages with images in this bundle; run on a bundle that has image messages (e.g. bundles_multimodal[1]).")